<a href="https://colab.research.google.com/github/devl2/hackathon2026/blob/main/hahahahahahackaton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Для запуска программы необходимо, чтобы датасет был на гугл драйвере в папке DATASET (или заменить в запуске path на тот, где находится датасет)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
!pip install pandas openpyxl xlrd

!pip install pdfplumber

!pip install python-docx

!pip install ijson

!pip install pytesseract pillow

!pip install pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 639.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 1.7 MB/s eta 0:00:00


In [5]:
import re
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import pandas as pd
import time

TEXT_EXT = {'.txt', '.csv', '.json', '.log', '.md'}

RE_EMAIL = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
RE_PHONE = re.compile(r'(?:\+7|8)[\s\-\(]?\d{3}[\s\-\)]?\d{3}[\s\-]?\d{2}[\s\-]?\d{2}')
RE_FIO = re.compile(r'[А-Я][а-я]+ [А-Я][а-я]+ [А-Я][а-я]+')
RE_SNILS = re.compile(r'\d{3}-\d{3}-\d{3} \d{2}')
RE_PASSPORT = re.compile(r'\d{4} \d{6}')
RE_INN = re.compile(r'\d{12}')
RE_CARD16 = re.compile(r'\d{16}')

ALL_PATTERNS = [
    (RE_EMAIL, 'EMAIL'),
    (RE_PHONE, 'PHONE'),
    (RE_FIO, 'FIO'),
    (RE_SNILS, 'SNILS'),
    (RE_PASSPORT, 'PASSPORT'),
    (RE_INN, 'INN'),
]

def fast_read(path: Path, max_chars=10000):
    try:
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read(max_chars)
    except:
        return ""

def get_level(pdn):
    total = sum(pdn.values())
    has_gov = any(k in pdn for k in ('PASSPORT','SNILS','INN'))
    has_pay = 'BANK_CARD' in pdn
    has_reg = any(k in pdn for k in ('EMAIL','PHONE','FIO'))

    if has_gov and total > 100:
        return 2
    if has_pay:
        return 2
    if has_gov:
        return 3
    if has_reg and total > 1000:
        return 3
    if has_reg:
        return 4
    return 0

def process_file(path: str, root: str):
    path = Path(path)
    root = Path(root)

    try:
        if path.suffix.lower() not in TEXT_EXT:
            return None

        text = fast_read(path)
        if not text:
            return None

        found = {}

        for pattern, key in ALL_PATTERNS:
            c = len(pattern.findall(text))
            if c:
                found[key] = c

        cards = RE_CARD16.findall(text)
        if cards:
            found['BANK_CARD'] = len(cards)

        if not found:
            return None

        return {
            "name": str(path.relative_to(root)),
            "path": str(path),
            "pdn": ', '.join(found.keys()),
            "count": sum(found.values()),
            "level": get_level(found)
        }

    except:
        return None

def walk_files(root: Path):
    for p in root.rglob("*"):
        if p.is_file():
            yield p

def scan(root: Path):
    start = time.time()

    files = list(walk_files(root))  # быстрее, чем pipeline
    results = []

    with ProcessPoolExecutor(max_workers=4) as pool:
        futures = [
            pool.submit(process_file, str(f), str(root))
            for f in files
        ]

        for i, fut in enumerate(as_completed(futures), 1):
            res = fut.result()
            if res:
                results.append(res)

            if i % 200 == 0:
                print(f"[Processed] {i}")

    print(f"Готово за {time.time()-start:.1f}s")
    return results


if __name__ == "__main__":
    root = Path("/content/drive/MyDrive/DATASET")

    results = scan(root)

    if results:
        df = pd.DataFrame(results).sort_values("level", ascending=False)
        df.to_csv("result.csv", index=False)
        print("Сохранено result.csv")
    else:
        print("Ничего не найдено")

[Processed] 200
[Processed] 400
[Processed] 600
[Processed] 800
[Processed] 1000
[Processed] 1200
[Processed] 1400
[Processed] 1600
[Processed] 1800
[Processed] 2000
[Processed] 2200
[Processed] 2400
[Processed] 2600
[Processed] 2800
[Processed] 3000
[Processed] 3200
Готово за 3.5s
Сохранено result.csv
